# YOLOv8 on the Russian Traffic Sign Dataset (RTSD)

Train a COCO-pretrained YOLOv8 detector (`yolov8n.pt` by default) on 30 RTSD classes.

The **Ultralytics** framework handles training, validation, and inference via `.train()`, `.val()`, and `.predict()`.


## 1. Install dependencies


In [ ]:
!pip install ultralytics pycocotools --quiet


In [ ]:
!kaggle datasets download -d watchman/rtsd-dataset

In [ ]:
!unzip -q rtsd-dataset.zip -d rtsd_data

## 2. Imports


In [ ]:
import os, json, random, shutil, pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict, Counter
from PIL import Image

from ultralytics import YOLO
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")


In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import random
OUTPUT_DIR = 'rtsd_yolo'
# Compare noisy test images.
gaussian_dir    = os.path.join(OUTPUT_DIR, 'images', 'test_gaussian')
saltpepper_dir  = os.path.join(OUTPUT_DIR, 'images', 'test_salt_pepper')

# Pick an image.
files = os.listdir(gaussian_dir)

fname = random.choice(files)

img_gaussian   = cv2.imread(os.path.join(gaussian_dir, fname))
img_saltpepper = cv2.imread(os.path.join(saltpepper_dir, fname))

img_gaussian   = cv2.cvtColor(img_gaussian,   cv2.COLOR_BGR2RGB)
img_saltpepper = cv2.cvtColor(img_saltpepper, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(img_gaussian)
axes[0].set_title('Gaussian noise', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_saltpepper)
axes[1].set_title('Salt-and-pepper noise', fontsize=12)
axes[1].axis('off')

plt.suptitle(f'Noisy image examples ({fname})', fontsize=11)
plt.tight_layout()
plt.savefig('noise_examples.png', dpi=150)
plt.show()
print(f'Image: {fname}')

## 3. Paths and training settings


In [ ]:
import os

# Dataset paths.
DATASET_ROOT = 'rtsd_data'

TRAIN_ANNO  = os.path.join(DATASET_ROOT, 'train_anno.json')
VAL_ANNO    = os.path.join(DATASET_ROOT, 'val_anno.json')
LABEL_MAP   = os.path.join(DATASET_ROOT, 'label_map.json')
FRAMES_DIR  = os.path.join(DATASET_ROOT, 'rtsd-frames', 'rtsd-frames')

# Training settings.
TOP_N                = 30
MAX_IMAGES_PER_CLASS = 1000
BATCH_SIZE           = 8
NUM_EPOCHS           = 15
IMG_SIZE             = 640
LR                   = 0.005
MODEL_NAME           = 'yolov8n.pt'   # Change to another YOLOv8 variant if needed.

YOLO_DATASET_DIR = 'rtsd_yolo'        # YOLO-formatted dataset.

print("Dataset paths configured.")
print(f"Dataset directory:   {DATASET_ROOT}")
print(f"Image directory exists: {os.path.exists(FRAMES_DIR)}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 4. Dataset analysis and class filtering


In [ ]:
# Load annotations.
with open(TRAIN_ANNO, 'r') as f:
    train_data = json.load(f)

with open(VAL_ANNO, 'r') as f:
    val_data = json.load(f)

with open(LABEL_MAP, 'r') as f:
    label_map = json.load(f)

print(f"Train: {len(train_data['images'])} images, {len(train_data['annotations'])} objects")
print(f"Val:   {len(val_data['images'])} images, {len(val_data['annotations'])} objects")
print(f"Classes in label_map: {len(label_map)}")

# Count class instances.
class_counts = Counter()
for ann in train_data['annotations']:
    class_counts[ann['category_id']] += 1

print(f"\nClasses in training data: {len(class_counts)}")
print(f"Max objects per class: {max(class_counts.values())}")
print(f"Min objects per class:  {min(class_counts.values())}")

# Select the most frequent classes.
top_classes   = {cat_id for cat_id, _ in class_counts.most_common(TOP_N)}
# Map class IDs to 0-based labels.
cat_id_to_new = {old: new for new, old in enumerate(sorted(top_classes))}
new_to_cat_id = {v: k for k, v in cat_id_to_new.items()}
NUM_CLASSES   = len(top_classes)
print(f"\nTop {TOP_N} classes: {sorted(top_classes)}")
print(f"NUM_CLASSES: {NUM_CLASSES}")

# Filter annotations.
train_anns_filtered = [a for a in train_data['annotations'] if a['category_id'] in top_classes]
train_img_ids       = {a['image_id'] for a in train_anns_filtered}
train_imgs_filtered = [img for img in train_data['images'] if img['id'] in train_img_ids]

# Limit images per class.
cat_to_imgs = defaultdict(list)
for ann in train_anns_filtered:
    cat_to_imgs[ann['category_id']].append(ann['image_id'])

imgs_to_keep = set()
for cat_id, img_ids in cat_to_imgs.items():
    unique_imgs = list(dict.fromkeys(img_ids))
    imgs_to_keep.update(unique_imgs[:MAX_IMAGES_PER_CLASS])

train_imgs_filtered = [img for img in train_imgs_filtered if img['id'] in imgs_to_keep]
train_anns_filtered = [a   for a   in train_anns_filtered if a['image_id'] in imgs_to_keep]

print(f"\nAfter filtering (max {MAX_IMAGES_PER_CLASS} images per class):")
print(f"Train: {len(train_imgs_filtered)} images, {len(train_anns_filtered)} objects")

# Plot class distribution.
class_counts_final = Counter(ann['category_id'] for ann in train_anns_filtered)
sorted_final = sorted(class_counts_final.items(), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar([str(x[0]) for x in sorted_final], [x[1] for x in sorted_final], color='darkorange')
ax.axhline(y=MAX_IMAGES_PER_CLASS, color='red', linestyle='--', label=f'Max = {MAX_IMAGES_PER_CLASS}')
ax.set_title(f'Filtered dataset — top {TOP_N} classes', fontsize=13)
ax.set_xlabel('category_id')
ax.set_ylabel('Number of objects')
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('class_distribution_final.png', dpi=100)
plt.show()


## 5. Convert annotations to YOLO format

Expected directory structure:
```
rtsd_yolo/
  images/train/  images/val/  images/test/
  labels/train/  labels/val/  labels/test/
```
Each image has a `.txt` label file containing normalized `class cx cy w h` values.


In [ ]:
import shutil

def write_yolo_labels(imgs_list, anns_list, frames_dir, img_out_dir, lbl_out_dir, cat_id_to_new):
    """
    Copy images and write YOLO labels.
    Convert COCO boxes to normalized YOLO coordinates.
    """
    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(lbl_out_dir, exist_ok=True)

    # Group annotations by image.
    ann_by_img = defaultdict(list)
    for ann in anns_list:
        ann_by_img[ann['image_id']].append(ann)

    copied = 0
    for img_info in imgs_list:
        img_id   = img_info['id']
        fname    = os.path.basename(img_info['file_name'])
        src_path = os.path.join(frames_dir, fname)
        dst_img  = os.path.join(img_out_dir, fname)
        dst_lbl  = os.path.join(lbl_out_dir, os.path.splitext(fname)[0] + '.txt')

        if not os.path.exists(src_path):
            continue

        # Image dimensions.
        W, H = img_info['width'], img_info['height']

        lines = []
        for ann in ann_by_img[img_id]:
            new_cls = cat_id_to_new.get(ann['category_id'])
            if new_cls is None:
                continue
            x, y, w, h = ann['bbox']
            if w <= 0 or h <= 0:
                continue
            # Normalize box coordinates.
            cx = (x + w / 2) / W
            cy = (y + h / 2) / H
            nw = w / W
            nh = h / H
            lines.append(f"{new_cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        if not lines:
            continue  # Skip images without selected classes.

        shutil.copy2(src_path, dst_img)
        with open(dst_lbl, 'w') as f:
            f.write('\n'.join(lines))
        copied += 1

    return copied


# Split images 70/15/15.
random.seed(42)
all_imgs = train_imgs_filtered.copy()
random.shuffle(all_imgs)

n_test = int(len(all_imgs) * 0.15)
n_val  = int(len(all_imgs) * 0.15)

test_imgs_split  = all_imgs[:n_test]
val_imgs_split   = all_imgs[n_test : n_test + n_val]
train_imgs_split = all_imgs[n_test + n_val:]

print(f"Train: {len(train_imgs_split)} | Val: {len(val_imgs_split)} | Test: {len(test_imgs_split)}")

# Assign annotations to each split.
train_ids = {img['id'] for img in train_imgs_split}
val_ids   = {img['id'] for img in val_imgs_split}
test_ids  = {img['id'] for img in test_imgs_split}

train_anns_split = [a for a in train_anns_filtered if a['image_id'] in train_ids]
val_anns_split   = [a for a in train_anns_filtered if a['image_id'] in val_ids]
test_anns_split  = [a for a in train_anns_filtered if a['image_id'] in test_ids]

# Write images and labels.
for split, imgs, anns in [
    ('train', train_imgs_split, train_anns_split),
    ('val',   val_imgs_split,   val_anns_split),
    ('test',  test_imgs_split,  test_anns_split),
]:
    n = write_yolo_labels(
        imgs, anns, FRAMES_DIR,
        img_out_dir=f'{YOLO_DATASET_DIR}/images/{split}',
        lbl_out_dir=f'{YOLO_DATASET_DIR}/labels/{split}',
        cat_id_to_new=cat_id_to_new,
    )
    print(f"{split}: saved {n} images")


## 6. Create the Ultralytics YAML configuration


In [ ]:
import yaml

# Use original category IDs as class names.

class_names = {new_id: str(old_id) for old_id, new_id in cat_id_to_new.items()}
names_list  = [class_names[i] for i in range(NUM_CLASSES)]

dataset_yaml = {
    'path':  os.path.abspath(YOLO_DATASET_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    NUM_CLASSES,
    'names': names_list,
}

yaml_path = os.path.join(YOLO_DATASET_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, allow_unicode=True, default_flow_style=False)

print(f"YAML saved: {yaml_path}")
print(f"nc = {NUM_CLASSES}")
print(f"Classes: {names_list}")


## 7. Inspect the training annotations


In [ ]:
import glob

# Select a training image.
sample_imgs = sorted(glob.glob(f'{YOLO_DATASET_DIR}/images/train/*.jpg'))[:1]
if not sample_imgs:
    sample_imgs = sorted(glob.glob(f'{YOLO_DATASET_DIR}/images/train/*.png'))[:1]

img_path = sample_imgs[0]
lbl_path = img_path.replace('images', 'labels').rsplit('.', 1)[0] + '.txt'

img = Image.open(img_path)
W, H = img.size

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(img)

with open(lbl_path) as f:
    for line in f:
        cls, cx, cy, nw, nh = map(float, line.split())
        x1 = (cx - nw/2) * W
        y1 = (cy - nh/2) * H
        bw = nw * W
        bh = nh * H
        rect = patches.Rectangle((x1, y1), bw, bh,
                                   linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-4, f'cls={int(cls)}', color='red', fontsize=8)

ax.set_title(f"Training example: {os.path.basename(img_path)}")
ax.axis('off')
plt.tight_layout()
plt.show()


## 8. Load COCO-pretrained YOLOv8

Ultralytics loads pretrained weights and adapts the detection head to the dataset's number of classes.


In [ ]:
# Load COCO-pretrained weights.
model = YOLO(MODEL_NAME)

print(f"Model: {MODEL_NAME}")
print(f"NUM_CLASSES: {NUM_CLASSES}")


## 9. Train the model


In [ ]:
results = model.train(
    data      = yaml_path,
    epochs    = NUM_EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH_SIZE,
    lr0       = LR,
    momentum  = 0.9,
    weight_decay = 0.0005,
    device    = device,
    project   = 'runs/rtsd',
    name      = 'yolov8_rtsd',
    exist_ok  = True,
    plots     = True,    # Save plots.
    save      = True,    # Save checkpoints.
    verbose   = True,
)

print("\nTraining completed!")
print(f"Best model: runs/rtsd/yolov8_rtsd/weights/best.pt")


## 10. Evaluate on the test set


In [ ]:
# Load the best checkpoint.
best_model = YOLO('runs/detect/runs/rtsd/yolov8_rtsd/weights/best.pt')

# Evaluate on the test split.
test_metrics = best_model.val(
    data   = yaml_path,
    split  = 'test',
    imgsz  = IMG_SIZE,
    device = device,
    plots  = True,
    save_json = True,   # Export predictions as JSON.
)

print("\nTest metrics:")
print(f"  mAP@0.50:0.95 : {test_metrics.box.map:.4f}")
print(f"  mAP@0.50      : {test_metrics.box.map50:.4f}")
print(f"  mAP@0.75      : {test_metrics.box.map75:.4f}")
print(f"  Precision     : {test_metrics.box.mp:.4f}")
print(f"  Recall        : {test_metrics.box.mr:.4f}")
print(f"\nPer-class AP50: {test_metrics.box.ap50}")


## 11. Training and evaluation plots

Ultralytics saves plots to `runs/rtsd/yolov8_rtsd/`. Display existing plots below.


In [ ]:
from IPython.display import Image as IPImage, display

plots_dir = 'runs/rtsd/yolov8_rtsd'
for plot_name in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    plot_path = os.path.join(plots_dir, plot_name)
    if os.path.exists(plot_path):
        print(f"\n{plot_name}:")
        display(IPImage(plot_path))
    else:
        print(f"{plot_name} not found — check the output path")


## 12. Save training history


In [ ]:
import pandas as pd

# Load training history.
results_csv = 'runs/rtsd/yolov8_rtsd/results.csv'
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print("History columns:", df.columns.tolist())
    print(df.tail())

    # Export the history to pickle.
    history = df.to_dict(orient='list')
    with open('history_rtsd_yolo.pickle', 'wb') as f:
        pickle.dump(history, f)
    print("\nHistory saved to history_rtsd_yolo.pickle")

# Save the class ID mapping.
with open('cat_id_to_new_yolo.json', 'w') as f:
    json.dump({str(k): v for k, v in cat_id_to_new.items()}, f)

print("Class mapping saved to cat_id_to_new_yolo.json")
print(f"Best model: runs/rtsd/yolov8_rtsd/weights/best.pt")


## 13. Visualize predictions


In [ ]:
import glob

def predict_and_show(model, img_path, gt_boxes_xywh, gt_labels,
                     device='cuda', threshold=0.5, nms_iou=0.3, cat_id_to_new=None):
    """
    Show ground truth and YOLOv8 predictions side by side.
    Ground-truth boxes use COCO pixel coordinates.
    """
    model.overrides['conf'] = threshold
    model.overrides['iou']  = nms_iou

    results = model.predict(img_path, device=device, verbose=False)[0]

    img = Image.open(img_path)
    W, H = img.size
    img_np = np.array(img)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Ground truth.
    axes[0].imshow(img_np)
    new_to_cat = {v: k for k, v in cat_id_to_new.items()} if cat_id_to_new else {}
    for (x, y, w, h), lbl in zip(gt_boxes_xywh, gt_labels):
        rect = patches.Rectangle((x, y), w, h,
                                   linewidth=2, edgecolor='green', facecolor='none')
        axes[0].add_patch(rect)
        orig = new_to_cat.get(lbl, lbl)
        axes[0].text(x, y-4, str(orig), color='green', fontsize=7)
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')

    # Predictions.
    axes[1].imshow(img_np)
    boxes  = results.boxes.xyxy.cpu().numpy()
    scores = results.boxes.conf.cpu().numpy()
    labels = results.boxes.cls.cpu().numpy().astype(int)
    for (x1, y1, x2, y2), score, lbl in zip(boxes, scores, labels):
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='red', facecolor='none')
        axes[1].add_patch(rect)
        orig = new_to_cat.get(lbl, lbl)
        axes[1].text(x1, y1-4, f'{orig}:{score:.2f}', color='red', fontsize=7)
    axes[1].set_title(f'YOLOv8 (conf={threshold}) | detections: {len(boxes)}')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()


# Select test images.
ann_by_img = defaultdict(list)
for ann in test_anns_split:
    ann_by_img[ann['image_id']].append(ann)

sample_indices = [0, 10, 50, 100]

for idx in sample_indices:
    if idx >= len(test_imgs_split):
        continue
    img_info = test_imgs_split[idx]
    fname    = os.path.basename(img_info['file_name'])
    img_path = os.path.join(YOLO_DATASET_DIR, 'images', 'test', fname)
    if not os.path.exists(img_path):
        continue

    anns = ann_by_img[img_info['id']]
    gt_boxes  = [[a['bbox'][0], a['bbox'][1], a['bbox'][2], a['bbox'][3]] for a in anns
                  if a['category_id'] in top_classes]
    gt_labels = [cat_id_to_new[a['category_id']] for a in anns
                  if a['category_id'] in top_classes]

    print(f"\nExample {idx}: {fname}")
    predict_and_show(best_model, img_path, gt_boxes, gt_labels,
                     device=device, threshold=0.5, cat_id_to_new=cat_id_to_new)


## 14. Load a saved model


In [ ]:
def load_model(model_path, device='cuda'):
    """Load a YOLOv8 model from a .pt checkpoint."""
    m = YOLO(model_path)
    m.to(device)
    return m

# Example:
# model_loaded = load_model('runs/rtsd/yolov8_rtsd/weights/best.pt')
# results = model_loaded.predict('path/to/image.jpg', conf=0.5)
print("load_model is ready.")
print(f"NUM_CLASSES = {NUM_CLASSES}")
